# Manifold-Matching Autoencoders — Colab quick start

Clone the repo, install deps, train an MMAE, and render an animated GIF
showing **the latent space evolving every epoch alongside the reference
embedding it's trying to match**.

Tip: switch the runtime to GPU (Runtime → Change runtime type → GPU). MNIST
with the full 60k pool is much faster on GPU; CPU is fine for `spheres`,
`mammoth`, and small MNIST subsets.

## 1. Clone the repo and install dependencies

In [ ]:
import os
REPO = 'manifold-matching-autoencoders'
if not os.path.exists(REPO):
    !git clone https://github.com/laurent-cheret/manifold-matching-autoencoders.git
%cd $REPO
!pip install -q -r requirements.txt

## 2. Pick what to train

All knobs in one place. The defaults below are a good starting point
for `mammoth` with 3D latent (the most visually rewarding demo).

In [ ]:
DATASET     = 'mammoth'    # 'mnist' | 'fmnist' | 'spheres' | 'mammoth'
REGULARIZER = 'mmae'       # 'mmae' | 'none'
REFERENCE   = 'pca'        # 'pca' | 'umap' | 'tsne'
REF_DIM     = 2            # 2 or 3 (used for the side-by-side reference panel)
LATENT_DIM  = 2            # 2 or 3 (mammoth + LATENT_DIM=3 looks great)
MM_NORMALIZE = True        # True = z-score distances (scale-invariant)
                           # False = match raw Euclidean distances
BATCHNORM   = True        # BatchNorm1d between hidden layers (recommended)
LAM         = 1.0          # weight on the manifold-matching term
EPOCHS      = 60
BATCH_SIZE  = 256
LR          = 1e-3
N_SAMPLES   = None         # None = loader default (60k mnist, 50k mammoth)
                           # 0    = use the full dataset
                           # int  = cap to that many points
SEED        = 42

## 3. Train

`snapshot_every=1` saves the latent embedding after every epoch so we can
build the GIF below.

In [ ]:
from mmae import train_run

result = train_run(
    dataset=DATASET,
    regularizer=REGULARIZER,
    reference=REFERENCE,
    ref_dim=REF_DIM,
    lam=LAM,
    mm_normalize=MM_NORMALIZE,
    batchnorm=BATCHNORM,
    latent_dim=LATENT_DIM,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    seed=SEED,
    n_samples=N_SAMPLES,
    snapshot_every=1,
)

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt

h = result.history
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h['train_total'], label='train total')
ax.plot(h['val_total'],   label='val total')
if REGULARIZER == 'mmae':
    ax.plot(h['train_recon'], label='train recon', linestyle='--')
    ax.plot(h['train_mm'],    label='train mm',    linestyle='--')
ax.set_xlabel('epoch'); ax.set_ylabel('loss')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Final latent space — side-by-side with the reference

When `regularizer='mmae'` is on, `result.ref_test` holds the reference
embedding the model was trained to match. Passing it to `plot_latents`
renders the static reference (left) next to the trained latent (right) so
you can read off how well the AE picked up the target structure.

In [ ]:
from mmae.viz import plot_latents

_, z, y = result.snapshots[-1]
plot_latents(
    z, y,
    reference=result.ref_test,
    reference_y=result.bundle.y_test,
    reference_title=f'{REFERENCE} reference ({REF_DIM}D)',
    title=f'{DATASET} · latent ({LATENT_DIM}D)',
)
plt.show()

## 6. Per-epoch GIF (reference + evolving latent)

Same trick on the GIF: the left panel is the static reference, the right
panel updates each epoch.

In [ ]:
from mmae.viz import make_latent_gif
from IPython.display import Image

gif_path = f'latent_evolution_{DATASET}_{REGULARIZER}_{REFERENCE}{REF_DIM}.gif'
make_latent_gif(
    result.snapshots, gif_path, fps=8,
    reference=result.ref_test,
    reference_y=result.bundle.y_test,
    reference_title=f'{REFERENCE} reference ({REF_DIM}D)',
    title_prefix=f'{DATASET} · {REGULARIZER}',
)
Image(filename=gif_path)

## 7. (Bonus) Vanilla AE for comparison

Same architecture, no manifold-matching term. Compare the latent geometry
to the MMAE result above.

In [ ]:
vanilla = train_run(
    dataset=DATASET, regularizer='none',
    batchnorm=BATCHNORM,
    latent_dim=LATENT_DIM, epochs=EPOCHS,
    batch_size=BATCH_SIZE, lr=LR, seed=SEED,
    n_samples=N_SAMPLES, snapshot_every=1,
)
_, z_v, y_v = vanilla.snapshots[-1]
plot_latents(z_v, y_v, title=f'{DATASET} · vanilla AE')
plt.show()

## 8. (Bonus) Try raw-distance matching

By default the loss z-scores both pairwise distance matrices, making the
latent's *shape* match the reference but not its scale. Set
`mm_normalize=False` to anchor the latent to the same metric scale as the
reference. Note: raw mode usually wants a smaller `lam`, since the loss
magnitude is now in squared-distance units.

In [ ]:
raw = train_run(
    dataset=DATASET, regularizer='mmae',
    reference=REFERENCE, ref_dim=REF_DIM,
    mm_normalize=False, lam=0.1,        # smaller lam for raw distances
    batchnorm=BATCHNORM,
    latent_dim=LATENT_DIM, epochs=EPOCHS,
    batch_size=BATCH_SIZE, lr=LR, seed=SEED,
    n_samples=N_SAMPLES, snapshot_every=1,
)
_, z_r, y_r = raw.snapshots[-1]
plot_latents(
    z_r, y_r,
    reference=raw.ref_test, reference_y=raw.bundle.y_test,
    reference_title=f'{REFERENCE} reference ({REF_DIM}D)',
    title=f'{DATASET} · mmae raw distances',
)
plt.show()